# R3 결과 살펴보기 (재학습 없음)

이 노트북은 저장소에 커밋된 **작은 결과표만** 읽습니다. 세포 배열·GPU·서버 자산이 필요 없습니다(pandas, matplotlib만 필요).
저장소 루트에서 실행하세요: `jupyter nbconvert --to notebook --execute tutorials/anchored_distribution_r3/01_inspect_results.ipynb`

순서: ① 분포 신호 진단과 gate → ② 평균 예측기 선택 → ③ dev에서의 복원 규칙 비교 → ④ 동결 후 held-out 약물 평가 → ⑤ SNU-423.

ED = Energy distance(작을수록 실제 처리군 세포에 가까움). 평균을 고정한 방법은 표본을 자기 평균으로 중심화하므로 값이 0보다 작을 수 있습니다.
방법끼리의 **차이**를 읽으세요.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "reports").exists() else Path.cwd().parents[1]
T = ROOT / "reports/anchored_distribution_r3/tables"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("tables:", sorted(p.name for p in T.iterdir()))

## ① 진단 D1/D2와 gate

`G1`은 "정답 평균으로 옮긴 대조군"이 "같은 조건의 다른 well"보다 실제 처리군에서 얼마나 더 먼가입니다. 구간 하한이 0보다 커야 통과입니다.

In [ ]:
d = json.loads((T / "d1_summary.json").read_text())
keys = ["all.a", "all.c_raw", "all.e_raw", "all.M_e_raw_minus_c_raw", "multi.b", "multi.d_mm", "multi.e_mm",
        "multi.G1_H_b_minus_d_mm", "multi.G1_ratio_H_over_a_minus_d_mm", "multi.G2_e_mm_minus_d_mm", "all.b_half_minus_c_mm"]
g = pd.DataFrame([{**{"quantity": k}, **d[k]} for k in keys]).round(4)
print(g.to_string(index=False))
G1 = d["multi.G1_H_b_minus_d_mm"]; M = d["all.M_e_raw_minus_c_raw"]
print()
print("M  (metric separates real-real from shuffled):", "PASS" if M["lo"] > 0 else "FAIL")
print("G1 (shape headroom beyond replicate wells):   ", "PASS" if G1["lo"] > 0 else "FAIL  -> generator claim on hold")

## ② 공통 평균 예측기 (train 약물 내부 3-fold CV)

In [ ]:
print(pd.read_csv(T / "mean_model_cv.tsv", sep="\t")[["kind", "hp", "rmse_cond", "rmse_cond_folds"]].round(5).to_string(index=False))
print(json.loads((ROOT / "configs/anchored_distribution_r3/mean_model.json").read_text()))

## ③ dev: 비음수 복원 전 / v1 / v2

같은 평균 μ̂, 같은 세포 수, 같은 복원을 쓰는 기준선들입니다.

In [ ]:
def show(name, variants=("pre", "anchored", "anchored2")):
    R = pd.read_csv(T / name, sep="\t")
    R[["m", "variant"]] = R.method.str.split(":", expand=True)
    R = R[R.variant.isin(variants) | (R.variant == "native")]
    cols = ["method", "n_drugs", "n_conditions", "ed", "ed_lo", "ed_hi", "ed_pca50", "mean_rmse", "var_ratio", "zero_frac_mae", "cluster_tv"]
    print(R[cols].round(4).to_string(index=False))
    return R
dev = show("dev_c1_restoration.tsv")

## ④ 동결 후 held-out 약물 평가 (전체 적격 세포주)

재사용 공개 벤치마크에서의 동결 후 평가입니다. DDPM·CellFlow는 gate에 따라 학습하지 않았으므로 행이 없습니다.

In [ ]:
fin = show("final_all_lines.tsv")
ref = "shifted:anchored2"
c = [k for k in fin.columns if k.startswith("d_ed_vs_" + ref)]
print()
print("paired difference vs shifted control (frozen restoration v2), drug-block bootstrap:")
print(fin[fin.variant == "anchored2"][["method"] + c].round(4).to_string(index=False))

## ⑤ SNU-423 (사전 지정 HCC 적용 대상, 별도 보고)

In [ ]:
snu = show("final_snu423.tsv", variants=("anchored2",))
c = [k for k in snu.columns if k.startswith("d_ed_vs_shifted:anchored2")]
print(snu[snu.variant == "anchored2"][["method"] + c].round(4).to_string(index=False))

In [ ]:
from IPython.display import Image, display
for f in sorted((ROOT / "reports/anchored_distribution_r3/figures").glob("*.png")):
    print(f.name); display(Image(filename=str(f), width=760))